In [9]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
from utils.reproducibility_utils import seed_everything
from utils.data_loadingbins import load_data, load_test_data

# --- 1. SETTINGS & PATHS ---
TRAIN_PATH = '../../datasets_challenge/train'
DIST_CSV = '../../datasets_challenge/sideSlip_distribution.csv'
MODEL_DIR = './expert_models'
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42) # Use your existing reproducibility util

In [10]:
class RNN(nn.Module):
    """
    Recurrent Neural Network for sideslip angle prediction.
    
    Architecture:
    1. RNN Layer - Processes sequential input
    2. Fully Connected Layer - Transforms RNN output
    3. Output Layer - Produces final prediction
    
    Dropout layers are included to prevent overfitting.
    """
    def __init__(self, input_size, output_size=1, rnn_units=28, fc_units=28, 
                 activation_fct=nn.ReLU(), dropout=[0.0, 0, 0], rnn_nonlinearity='relu'):
        super(RNN, self).__init__()
        
        # RNN layer: processes the sequence and captures temporal dependencies
        self.rnn_layer = nn.RNN(input_size=input_size, hidden_size=rnn_units, 
                                num_layers=1, nonlinearity=rnn_nonlinearity, batch_first=True)
        
        # Fully connected layers for processing RNN output
        self.fc = nn.Linear(rnn_units, fc_units)
        self.output_layer = nn.Linear(fc_units, output_size)
        
        # Dropout layers to prevent overfitting (randomly zeros out neurons during training)
        self.dropout1 = nn.Dropout(dropout[0])  # Applied to input
        self.dropout2 = nn.Dropout(dropout[1])  # Applied after RNN
        self.dropout3 = nn.Dropout(dropout[2])  # Applied after FC layer
        
        self.activation_fct = activation_fct
 
    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        x = self.dropout1(x)
        
        # Process sequence through RNN
        x, _ = self.rnn_layer(x)  # Output shape: (batch_size, sequence_length, rnn_units)
        
        # Take only the last time step's output (t0)
        x = x[:, -1, :]  # Shape: (batch_size, rnn_units)
        
        x = self.dropout2(x)
        x = self.activation_fct(self.fc(x))  # Apply fully connected layer with activation
        x = self.dropout3(x)
        x = self.output_layer(x)  # Final prediction
        
        return x

In [11]:
from sklearn.metrics import mean_absolute_error

device = "cuda" if torch.cuda.is_available() else "cpu"

model = RNN(
    input_size=22,          # number of features
    output_size=1,
    rnn_units=48,
    fc_units=48,
    dropout=[0.0, 0.0, 0.0]
).to(device)

criterion = nn.L1Loss()     # MAE (matches few-shot objective)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)


In [12]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [13]:
def validate(model, dataloader, device):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            pred = model(X).squeeze(1)
            preds.append(pred.cpu().numpy())
            targets.append(y.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    return mean_absolute_error(targets, preds)


In [16]:
print("Loading train data...")
train_loader, val_loader, pipeline = load_data(
            train_path=TRAIN_PATH,
            dist_csv_path=DIST_CSV,
            val_file_name="train_part_017.csv", 
            columns_to_drop=["sideSlip", "vyCG", "time", "longitude","latitude"],
            sequence_length=20,
            threshold=True,
            threshold_value=3.0,
            threshold_column="vxCG",
            target='sideSlip',
            batchsize=64
        )

Loading train data...
VALIDATION: Loading train_part_017.csv
Calculating weights for WeightedRandomSampler...


In [17]:
EPOCHS = 25
best_val_mae = float("inf")
best_model_state = None

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model,
        train_loader,   # ← weighted sampler is used automatically
        optimizer,
        criterion,
        device
    )

    val_mae = validate(
        model,
        val_loader,     # ← no sampler, true distribution
        device
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val MAE: {val_mae:.4f}"
    )

    # Save best model (important for snapshot / ensemble)
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_model_state = model.state_dict()


Epoch 01 | Train Loss: 0.3882 | Val MAE: 0.3575
Epoch 02 | Train Loss: 0.2449 | Val MAE: 0.4396
Epoch 03 | Train Loss: 0.2211 | Val MAE: 0.3762
Epoch 04 | Train Loss: 0.2077 | Val MAE: 0.4223
Epoch 05 | Train Loss: 0.2015 | Val MAE: 0.3323
Epoch 06 | Train Loss: 0.1955 | Val MAE: 0.3186
Epoch 07 | Train Loss: 0.1904 | Val MAE: 0.3164
Epoch 08 | Train Loss: 0.1894 | Val MAE: 0.3426
Epoch 09 | Train Loss: 0.1835 | Val MAE: 0.3474
Epoch 10 | Train Loss: 0.1815 | Val MAE: 0.2985
Epoch 11 | Train Loss: 0.1797 | Val MAE: 0.3805
Epoch 12 | Train Loss: 0.1781 | Val MAE: 0.3413
Epoch 13 | Train Loss: 0.1766 | Val MAE: 0.4263
Epoch 14 | Train Loss: 0.1751 | Val MAE: 0.3836
Epoch 15 | Train Loss: 0.1717 | Val MAE: 0.2496
Epoch 16 | Train Loss: 0.1720 | Val MAE: 0.2479
Epoch 17 | Train Loss: 0.1715 | Val MAE: 0.3469
Epoch 18 | Train Loss: 0.1709 | Val MAE: 0.3665
Epoch 19 | Train Loss: 0.1683 | Val MAE: 0.3485
Epoch 20 | Train Loss: 0.1664 | Val MAE: 0.3633
Epoch 21 | Train Loss: 0.1667 | Val MAE:

In [18]:
model.load_state_dict(best_model_state)
print(f"✓ Best validation MAE: {best_val_mae:.4f}")


✓ Best validation MAE: 0.2479


In [19]:
print("\nLoading test data...")
test_dataloader = load_test_data(test_path = "../../datasets_challenge/test",
                                 batchsize = 64,
                                 pipeline = pipeline, 
                                 load_anonymized_data=True)


Loading test data...
../../datasets_challenge/test\test_part_001.csv
../../datasets_challenge/test\test_part_002.csv
../../datasets_challenge/test\test_part_003.csv
../../datasets_challenge/test\test_part_004.csv
../../datasets_challenge/test\test_part_005.csv
../../datasets_challenge/test\test_part_006.csv
../../datasets_challenge/test\test_part_007.csv
../../datasets_challenge/test\test_part_008.csv
../../datasets_challenge/test\test_part_009.csv
../../datasets_challenge/test\test_part_010.csv
../../datasets_challenge/test\test_part_011.csv


In [20]:
# Generate predictions on test set
model.eval()
total_loss = 0  
test_preds = []
test_ground_truths = []  # Will contain placeholder values (all zeros)

with torch.no_grad():
    for step, (x, y) in enumerate(test_dataloader):
        X, y = x.to(device), y.to(device).reshape([len(y), 1])

        pred = model(X)
        loss = criterion(pred, y)
        # Note: loss is meaningless here since y contains placeholder values
        
        # Store predictions
        test_preds.append(pred.reshape(-1).cpu().numpy())
        test_ground_truths.append(y.reshape(-1).cpu().numpy())

    # Concatenate all batches
    test_ground_truths = np.concatenate(test_ground_truths)
    test_preds = np.concatenate(test_preds)
    
print("Shape of Predictions", test_preds.shape)

Shape of Predictions (49328,)


In [22]:
# Save predictions in the required format for submission
y_pred = pd.DataFrame({"predicted_sideslip": test_preds})
y_pred["id"] = range(len(test_preds))  # Add sample IDs (0 to 49327)

# Save to CSV file
y_pred.to_csv("predictions09.csv", index=False)

print(f"✓ Predictions saved to 'predictions09.csv'")
print(f"✓ Total samples: {len(test_preds)}")
print(f"✓ Ready for submission to the competition!")

✓ Predictions saved to 'predictions09.csv'
✓ Total samples: 49328
✓ Ready for submission to the competition!
